In [1]:
import pandas as pd
import sqlite3
import plotly.graph_objects as go

In [2]:
pd.set_option('display.max_columns', None)

## Ex09
Сделай всё необходимое, чтобы получить график как на примере.

In [3]:
db_path = "../data/checking-logs.sqlite"
conn = sqlite3.connect(db_path)

In [4]:
commits = pd.io.sql.read_sql(
    """
    SELECT uid, timestamp, numTrials
    FROM checker
    WHERE uid LIKE 'user_%'
      AND status = 'ready'
      AND labname = 'project1'
    ORDER BY timestamp
    """,
    conn,
    parse_dates=["timestamp"]
)

commits['timestamp'] = commits['timestamp'].dt.date
commits.head()

,uid,timestamp,numTrials
0,user_4,2020-04-17,1
1,user_4,2020-04-17,2
2,user_4,2020-04-17,3
3,user_4,2020-04-17,4
4,user_4,2020-04-17,5


In [5]:
days = commits.groupby('timestamp')['numTrials'].max().reset_index()
days['day'] = days.index.values
commits = commits.merge(days, on='timestamp', suffixes=('', '_y'))
commits.head()

,uid,timestamp,numTrials,numTrials_y,day
0,user_4,2020-04-17,1,7,0
1,user_4,2020-04-17,2,7,0
2,user_4,2020-04-17,3,7,0
3,user_4,2020-04-17,4,7,0
4,user_4,2020-04-17,5,7,0


In [6]:
commits = commits.groupby(['uid', 'day'])['numTrials'].max().rename('max').reset_index()
commits.head()

,uid,day,max
0,user_1,17,11
1,user_10,15,7
2,user_10,16,21
3,user_10,17,59
4,user_11,6,1


In [7]:
pivot = commits.pivot(index='uid', columns='day', values='max')
pivot = pivot.ffill(axis=1)
pivot = pivot.fillna(0)
pivot = pivot.astype(int)
pivot.head()

day,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18
uid,,,,,,,,,,,,,,,,,,,
user_1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,11,11
user_10,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,7,21,59,59
user_11,0,0,0,0,0,0,1,1,1,1,1,1,1,1,1,1,1,1,1
user_12,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,4,4
user_13,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,30,30,32,32


In [8]:
initial_data = [
    go.Scatter(
        x=pivot.columns,
        y=pivot.loc[uid],
        name=uid,
        mode='lines+markers'
    )
    for uid in pivot.index
]

In [9]:
frames = []
for i in range(1, len(pivot.columns) + 1):
    frame_data = [
        go.Scatter(
            x=pivot.columns[:i],
            y=pivot.loc[uid][:i],
            mode='lines+markers',
            name=uid
        )
        for uid in pivot.index
    ]

    frames.append(go.Frame(data=frame_data, name=str(i)))

In [10]:
xa = len(pivot.columns)
ya = pivot.values.max() + 2

layout = go.Layout(
    title='Dynamic of commits per user in project1',
    xaxis=dict(range=(0, xa)),
    yaxis=dict(range=(0, ya)),
    height=700,
    updatemenus=[{
        'type': 'buttons',
        'buttons': [{'method': 'animate', 'label': 'play', 'args': [None]}]
    }]
)

figure = go.Figure(data=initial_data, layout=layout, frames=frames)
figure.show()

In [11]:
conn.close()